# Energy Prices vs Inflation Analysis

What kind of relationship do gasoline, crude oil, and natural gas prices have with inflation - and how do their prices themselves behave through the year?

The notebook has two parts:

1. **Setup** - load the data, clean it, merge it, and plot the overall trends.
2. **Analysis** Energy prices vs. the inflation rate (with lead/lag). Convert the CPI index into the year-over-year inflation rate, correlate it with each energy price, and check if energy prices *lead* inflation by some number of months.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


### First I'll read the CSV files and store them in separate dataframes.

In [ ]:
gas = pd.read_csv("../data/raw/gas_prices.csv")
oil = pd.read_csv("../data/raw/crude_oil.csv")
natural = pd.read_csv("../data/raw/natural_gas.csv")
inflation = pd.read_csv("../data/raw/inflation.csv")

### Before doing analysis, I'll quickly check the first 5 rows of each dataset.

In [ ]:
print("Gas")
display(gas.head())

print("Oil")
display(oil.head())

print("Natural Gas")
display(natural.head())

print("Inflation")
display(inflation.head())

### Clean dates and keep only needed columns

Convert date columns into datetime format and simplify each dataframe.

In [ ]:
gas["period"] = pd.to_datetime(gas["period"])
oil["period"] = pd.to_datetime(oil["period"])
natural["date"] = pd.to_datetime(natural["date"])
inflation["date"] = pd.to_datetime(inflation["date"])

# FRED uses "." for missing values, so coerce to numeric
natural["value"] = pd.to_numeric(natural["value"], errors="coerce")
inflation["value"] = pd.to_numeric(inflation["value"], errors="coerce")

gas = gas[["period","value"]]
oil = oil[["period","value"]]
natural = natural[["date","value"]]
inflation = inflation[["date","value"]]

gas.columns = ["date","gas"]
oil.columns = ["date","oil"]
natural.columns = ["date","natural_gas"]
inflation.columns = ["date","cpi"]

### I'll join everything together by date so values appear side by side.

Because inflation data is monthly and the others are weekly, I'll make some changes so that all of them shows monthly data.

In [ ]:
# Convert all dates into month-year only
gas["date"] = gas["date"].dt.to_period("M")
oil["date"] = oil["date"].dt.to_period("M")
natural["date"] = natural["date"].dt.to_period("M")
inflation["date"] = inflation["date"].dt.to_period("M")

# Now I have 4-5 of values in the same month. Average all rows inside each of those months.
# (Natural gas and CPI are already monthly, so the average is just the single row.)
gas = gas.groupby("date")["gas"].mean().reset_index()
oil = oil.groupby("date")["oil"].mean().reset_index()
natural = natural.groupby("date")["natural_gas"].mean().reset_index()

df = gas.merge(oil, on="date")
df = df.merge(natural, on="date")
df = df.merge(inflation, on="date")

display(df.head())

### Plot Trends


In [ ]:
# Convert date if needed
if str(df["date"].dtype) == "period[M]":
    df["date"] = df["date"].dt.to_timestamp()
cols = ["gas", "oil", "natural_gas", "cpi"]
for col in cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df = df.dropna()


# This whole above protion is for sanity check coz' I've done this already before too other than the "cols" list.

scaled = df.copy()

# Normalize
for col in cols:
    scaled[col] = scaled[col] / scaled[col].iloc[0]

plt.figure(figsize=(14,6))

for col in cols:
    plt.plot(
        scaled["date"],
        scaled[col],
        label=col,
        alpha=0.6
    )

plt.xlabel("Date")
plt.ylabel("Normalized Value (start = 1.0)")
plt.title("Normalized Trends: Gas, Oil, Natural Gas, CPI")
plt.legend()

plt.show()

## Analysis - Energy Prices vs. the Inflation Rate (and Lead/Lag)

A quick note on the data: `inflation.csv` is actually the **CPI index level** (FRED series `CPIAUCSL`, with base period 1982–84 = 100). It is *not* the inflation rate. CPI grows almost monotonically, so a raw correlation of CPI vs energy prices doesn't really answer the question we care about. Thats why we need to convert the CPI into the year-over-year (YoY) inflation rate first.

Steps:

1. Convert CPI level to YoY inflation rate.
2. Plot energy prices alongside the inflation rate.
3. Correlate each energy price with the inflation rate.
4. Lead/lag check: shift the energy series back in time by 1, 2,..., 12 months and see which lag has the strongest correlation. A best-lag of, say, 3 months would mean energy prices today match inflation three months from now (i.e. energy leads inflation by 3 months).

### Step 1 - Build the YoY inflation rate

YoY inflation = how much CPI changed compared to the same month last year, expressed as a percentage:

```
inflation_rate(t) = (CPI(t) / CPI(t - 12 months) - 1) * 100
```

The first 12 months will be NaN (because there is no "12 months ago" value yet), so we drop them.

In [ ]:
df = df.sort_values("date").reset_index(drop=True)

# YoY inflation rate, in percent formula
df["inflation_rate"] = (df["cpi"] / df["cpi"].shift(12) - 1) * 100

# Drop the first 12 rows becuase there;s nothing there
df = df.dropna(subset=["inflation_rate"]).reset_index(drop=True)

display(df[["date", "cpi", "inflation_rate"]].head())

### Step 2 - Visualize inflation rate vs. energy prices

Twin y-axis plot: inflation rate on the right (%), energy prices on the left ($).

In [ ]:
fig, ax_left = plt.subplots(figsize=(14, 6))

# Energy prices on the left axis (oil is divided by 20 so it fits on the same scale)
ax_left.plot(df["date"], df["gas"], label="gas ($/gal)", alpha=0.7, color="tab:blue")
ax_left.plot(df["date"], df["oil"] / 20, label="oil ($/bbl / 20)", alpha=0.7, color="tab:orange")
ax_left.plot(df["date"], df["natural_gas"], label="natural gas ($/MMBTU)", alpha=0.7, color="tab:green")
ax_left.set_xlabel("Date")
ax_left.set_ylabel("Energy price")
ax_left.legend(loc="upper left")

# Inflation rate on a separate right axis
ax_right = ax_left.twinx()
ax_right.plot(df["date"], df["inflation_rate"], label="YoY inflation %", color="black", linewidth=2)
ax_right.axhline(0, linestyle="--", linewidth=0.5, color="gray")
ax_right.set_ylabel("YoY inflation rate (%)")
ax_right.legend(loc="upper right")

plt.title("Energy prices vs. YoY inflation rate")
plt.show()

### Correlation against the inflation rate 

In [ ]:
# How tightly do gas / oil / natural gas move with the YoY inflation rate?
correlations = df[["gas", "oil", "natural_gas", "inflation_rate"]].corr()
display(correlations)

print("\nCorrelation of each energy price with the YoY inflation rate:")
print(correlations["inflation_rate"].drop("inflation_rate").sort_values(ascending=False))

### Lead/lag analysis

The correlation above answers "do they move together this month?" But energy is an **input cost** - a gas price spike might take a few months to filter through transportation, packaging, food, and into the overall CPI basket.

For each lag `k = 0, 1, 2, ..., 12` months, we shift the energy series back by `k` months and correlate with today's inflation rate. The lag with the highest correlation is the one where energy "leads" inflation by that many months.

In [ ]:
# convert each energy series to its own YoY % change.
df["gas_yoy"]         = (df["gas"]         / df["gas"].shift(12)         - 1) * 100
df["oil_yoy"]         = (df["oil"]         / df["oil"].shift(12)         - 1) * 100
df["natural_gas_yoy"] = (df["natural_gas"] / df["natural_gas"].shift(12) - 1) * 100

# for each lag k = 0...12 months, correlate "energy YoY shifted back by k months"
energy_yoy_cols = ["gas_yoy", "oil_yoy", "natural_gas_yoy"]
lag_table = pd.DataFrame(index=range(0, 13), columns=energy_yoy_cols, dtype=float)
lag_table.index.name = "lag_months"

#I'm checking one by one, which month gives the best correlation with the inflation rate.
for col in energy_yoy_cols:
    for k in lag_table.index:
        lag_table.loc[k, col] = df[col].shift(k).corr(df["inflation_rate"])

display(lag_table.round(3))

print("\nBest lag for each energy series (the month where the correlation is highest):")
for col in energy_yoy_cols:
    best_lag = lag_table[col].idxmax()
    best_corr = lag_table[col].max()
    print(f"  {col:18s}  lag = {best_lag:2d} months   corr = {best_corr:+.3f}")

In [ ]:
plt.figure(figsize=(12, 5))

for col in energy_yoy_cols:
    plt.plot(lag_table.index, lag_table[col], marker="o", label=col)

plt.axhline(0, color="gray", linewidth=0.5)
plt.xlabel("Lag in months (energy series shifted back by this many months)")
plt.ylabel("Correlation with YoY inflation")
plt.title("Lead/lag: YoY energy price change vs. YoY inflation")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Analysis

### Correlation with Inflation (YoY)

| Variable | Correlation |
|-----------|------------|
| Oil | +0.69 |
| Gasoline | +0.63 |
| Natural Gas | -0.22 |

- Oil and gasoline have a strong positive relationship with inflation.
- Oil has the strongest correlation (+0.69).
- Natural gas does not follow inflation closely and shows a weak negative correlation (-0.22).

### Lead/Lag Results

| Variable | Best Lag | Correlation |
|-----------|----------|------------|
| Gasoline | 0 months | 0.91 |
| Oil | 0 months | 0.88 |
| Natural Gas | 12 months | 0.96 |

- Oil and gasoline have their strongest relationship with inflation at **0 months**, meaning they move at the same time as inflation.
- The 12-month natural gas result is likely caused by limited data and should not be treated as a real predictive effect.
- Overall, oil and gasoline track inflation closely, while natural gas does not.

### Notes

- Results are based on data from 2015–2026, which includes COVID-19 and the 2021–22 inflation spike.
- Correlations may look different over other time periods.
- Correlation does not prove causation.
- Other factors, such as interest rates, supply chains, and currency movements, were not included in this analysis.